In [ ]:
"""
Infer domain-squatting transformations and express them as Hashcat-style rules.

Input JSON format:
{
  "original.com": {
     "transformed1.com": <int>,
     "transformed2.com": <int>,
     ...
  },
  ...
}

We compare SLDs (second-level domains, excluding .com), infer a minimal
Damerau-Levenshtein edit script, then map edits to hashcat rules:

Mapping:
- delete first char            ->  [
- delete last char             ->  ]
- delete at position N         ->  D N
- prepend char X               ->  ^ X   (multi-char prefix becomes multiple ^ rules,
                                         applied in reverse order to preserve prefix)
- append char X                ->  $ X   (multi-char suffix becomes multiple $ rules)
- insert char X at position N  ->  i N X
- overwrite position N with X  ->  o N X
- swap positions i and i+1     ->  * i (i+1)      (for adjacent transposition)

Rules are emitted as space-separated tokens like best64.rule.
Multiple ops are combined in order into ONE composite rule string.

Outputs:
- Writes rule_counts.json
- Writes domain.rule sorted by frequency (desc, stable tie-break)

NOTE: Domain names are lowercased; case rules are not generated.
"""

from __future__ import annotations
import json
from dataclasses import dataclass
from collections import Counter
from typing import List, Tuple, Union, Iterable, Set

# ---------------------------
# base36 encoding/decoding
# ---------------------------
_BASE36 = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"
_BASE36_MAP = {ch: i for i, ch in enumerate(_BASE36)}

def enc_pos(n: int) -> str:
    if n < 0 or n >= len(_BASE36):
        raise ValueError(f"Position {n} out of encodable range 0-35")
    return _BASE36[n]

def dec_pos(ch: str) -> int:
    ch = ch.upper()
    if ch not in _BASE36_MAP:
        raise ValueError(f"Bad base36 position: {ch}")
    return _BASE36_MAP[ch]

# ---------------------------
# edit-script inference
# ---------------------------
@dataclass
class Op:
    kind: str   # 'ins', 'del', 'sub', 'trans', 'keep'
    i: int      # index in original (for ins, position before which inserted)
    j: int      # index in transformed
    a: str = '' # original char(s)
    b: str = '' # transformed char(s)

def sld(domain: str) -> str:
    return domain.split('.', 1)[0].lower()

def damerau_levenshtein_ops(a: str, b: str) -> List[Op]:
    """Minimal edit script with adjacent transpositions."""
    n, m = len(a), len(b)
    dp = [[0]*(m+1) for _ in range(n+1)]
    back = [[None]*(m+1) for _ in range(n+1)]

    for i in range(1, n+1):
        dp[i][0] = i
        back[i][0] = Op('del', i-1, 0, a[i-1], '')
    for j in range(1, m+1):
        dp[0][j] = j
        back[0][j] = Op('ins', 0, j-1, '', b[j-1])

    for i in range(1, n+1):
        for j in range(1, m+1):
            cost_sub = 0 if a[i-1] == b[j-1] else 1

            best_cost = dp[i-1][j] + 1
            best_op = Op('del', i-1, j, a[i-1], '')

            c_ins = dp[i][j-1] + 1
            if c_ins < best_cost:
                best_cost = c_ins
                best_op = Op('ins', i, j-1, '', b[j-1])

            c_sub = dp[i-1][j-1] + cost_sub
            if c_sub < best_cost:
                best_cost = c_sub
                best_op = Op('keep' if cost_sub==0 else 'sub', i-1, j-1, a[i-1], b[j-1])

            if i >= 2 and j >= 2 and a[i-2] == b[j-1] and a[i-1] == b[j-2]:
                c_trans = dp[i-2][j-2] + 1
                if c_trans < best_cost:
                    best_cost = c_trans
                    best_op = Op('trans', i-2, j-2, a[i-2:i], b[j-2:j])

            dp[i][j] = best_cost
            back[i][j] = best_op

    # backtrace
    ops: List[Op] = []
    i, j = n, m
    while i > 0 or j > 0:
        op = back[i][j]
        if op is None:
            break
        ops.append(op)
        if op.kind == 'del':
            i -= 1
        elif op.kind == 'ins':
            j -= 1
        elif op.kind in ('sub','keep'):
            i -= 1
            j -= 1
        elif op.kind == 'trans':
            i -= 2
            j -= 2
    ops.reverse()
    return [o for o in ops if o.kind != 'keep']

def ops_to_hashcat_tokens(orig: str, trans: str, ops: List[Op]) -> List[str]:
    """Convert edit ops to hashcat rule tokens."""
    n = len(orig)
    tokens: List[str] = []

    prefix_chars: List[str] = []
    suffix_chars: List[str] = []
    mid_inserts: List[Op] = []

    for op in ops:
        if op.kind == 'ins':
            if op.i == 0:
                prefix_chars.append(op.b)
            elif op.i == n:
                suffix_chars.append(op.b)
            else:
                mid_inserts.append(op)
        elif op.kind == 'del':
            if op.i == 0:
                tokens.append('[')
            elif op.i == n-1:
                tokens.append(']')
            else:
                tokens.append(f"D{enc_pos(op.i)}")
        elif op.kind == 'sub':
            tokens.append(f"o{enc_pos(op.i)}{op.b}")
        elif op.kind == 'trans':
            i0 = op.i
            tokens.append(f"*{enc_pos(i0)}{enc_pos(i0+1)}")

    # prefix inserts: reverse so final prefix order is correct
    for ch in reversed(prefix_chars):
        tokens.insert(0, f"^{ch}")

    # suffix inserts in order
    for ch in suffix_chars:
        tokens.append(f"${ch}")

    # middle inserts by increasing pos
    for op in sorted(mid_inserts, key=lambda x: x.i):
        tokens.append(f"i{enc_pos(op.i)}{op.b}")

    return tokens

def infer_rule(orig_domain: str, trans_domain: str) -> str:
    o = sld(orig_domain)
    t = sld(trans_domain)
    ops = damerau_levenshtein_ops(o, t)
    tokens = ops_to_hashcat_tokens(o, t, ops)
    return " ".join(tokens) if tokens else ":"


# ==========================================================
# Your own (non-hashcat) rule engine for further use
# ==========================================================
OpType = str
OpArgs = Tuple[Union[int, str], ...]
Rule = List[Tuple[OpType, OpArgs]]

def parse_domain_rule_line(line: str) -> Rule:
    """
    Parse one composite rule line (space-separated tokens) into ops.
    Supported tokens: :, [, ], Dn, ^c, $c, in c, on c, *ij  (n,i,j base36)
    """
    line = line.strip()
    if not line or line.startswith("#"):
        return []

    tokens = line.split()
    ops: Rule = []
    for tok in tokens:
        if tok == ":":
            ops.append((':', ()))
        elif tok == "[":
            ops.append(('[', ()))
        elif tok == "]":
            ops.append((']', ()))
        elif tok.startswith("D") and len(tok) == 2:
            ops.append(('D', (dec_pos(tok[1]),)))
        elif tok.startswith("^") and len(tok) == 2:
            ops.append(('^', (tok[1],)))
        elif tok.startswith("$") and len(tok) == 2:
            ops.append(('$', (tok[1],)))
        elif tok.startswith("i") and len(tok) == 3:
            ops.append(('i', (dec_pos(tok[1]), tok[2])))
        elif tok.startswith("o") and len(tok) == 3:
            ops.append(('o', (dec_pos(tok[1]), tok[2])))
        elif tok.startswith("*") and len(tok) == 3:
            ops.append(('*', (dec_pos(tok[1]), dec_pos(tok[2]))))
        else:
            raise ValueError(f"Unrecognized token: {tok}")
    return ops

def apply_domain_rule(word: str, rule: Rule) -> str:
    """Apply one parsed rule to a word (left-to-right)."""
    s = word
    for op, args in rule:
        if op == ":":
            continue
        elif op == "[":
            s = s[1:] if s else s
        elif op == "]":
            s = s[:-1] if s else s
        elif op == "D":
            (pos,) = args
            pos = int(pos)
            if 0 <= pos < len(s):
                s = s[:pos] + s[pos+1:]
        elif op == "^":
            (ch,) = args
            s = str(ch) + s
        elif op == "$":
            (ch,) = args
            s = s + str(ch)
        elif op == "i":
            pos, ch = args
            pos = int(pos)
            ch = str(ch)
            if pos < 0:
                pos = 0
            if pos > len(s):
                pos = len(s)
            s = s[:pos] + ch + s[pos:]
        elif op == "o":
            pos, ch = args
            pos = int(pos)
            ch = str(ch)
            if 0 <= pos < len(s):
                s = s[:pos] + ch + s[pos+1:]
        elif op == "*":
            i, j = map(int, args)
            if 0 <= i < len(s) and 0 <= j < len(s) and i != j:
                lst = list(s)
                lst[i], lst[j] = lst[j], lst[i]
                s = "".join(lst)
        else:
            raise RuntimeError(f"Unknown op: {op}")
    return s

def load_sorted_domain_rules(path: str = "./domain.rule") -> List[Rule]:
    """
    Load domain.rule (already sorted by frequency) and parse into our Rule objects.
    """
    rules: List[Rule] = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            rules.append(parse_domain_rule_line(line))
    return rules

def apply_domain_rules(word: str, rules: Iterable[Rule]) -> Set[str]:
    """Apply many rules and return all outputs."""
    out: Set[str] = set()
    for r in rules:
        try:
            out.add(apply_domain_rule(word, r))
        except Exception:
            pass
    return out

In [ ]:
with open('./disputes-training.json', "r", encoding="utf-8") as f:
    data = json.load(f)

counts = Counter()
for orig, trans_map in data.items():
    for trans in trans_map.keys():
        rule = infer_rule(orig, trans)
        counts[rule] += 1

with open('./rule_counts.json', "w", encoding="utf-8") as f:
    json.dump(counts, f, indent=2, sort_keys=True)

# sort by usage desc, stable tie-break
sorted_rules = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))

with open('./domain.rule', "w", encoding="utf-8") as f:
    for r, c in sorted_rules:
        f.write(r + "\n")
